<a href="https://colab.research.google.com/github/AI4Finance-Foundation/FinRL-Tutorials/blob/master/1-Introduction/Stock_NeurIPS2018_SB3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deep Reinforcement Learning for Stock Trading from Scratch: Multiple Stock Trading

* **Pytorch Version** 



# Content

* [1. Task Description](#0)
* [2. Install Python packages](#1)
    * [2.1. Install Packages](#1.1)    
    * [2.2. A List of Python Packages](#1.2)
    * [2.3. Import Packages](#1.3)
    * [2.4. Create Folders](#1.4)
* [3. Download and Preprocess Data](#2)
* [4. Preprocess Data](#3)        
    * [4.1. Technical Indicators](#3.1)
    * [4.2. Perform Feature Engineering](#3.2)
* [5. Build Market Environment in OpenAI Gym-style](#4)  
    * [5.1. Data Split](#4.1)  
    * [5.3. Environment for Training](#4.2)    
* [6. Train DRL Agents](#5)
* [7. Backtesting Performance](#6)  
    * [7.1. BackTestStats](#6.1)
    * [7.2. BackTestPlot](#6.2)   
  

<a id='0'></a>
# Part 1. Task Discription

We train a DRL agent for stock trading. This task is modeled as a Markov Decision Process (MDP), and the objective function is maximizing (expected) cumulative return.

We specify the state-action-reward as follows:

* **State s**: The state space represents an agent's perception of the market environment. Just like a human trader analyzing various information, here our agent passively observes many features and learns by interacting with the market environment (usually by replaying historical data).

* **Action a**: The action space includes allowed actions that an agent can take at each state. For example, a ∈ {−1, 0, 1}, where −1, 0, 1 represent
selling, holding, and buying. When an action operates multiple shares, a ∈{−k, ..., −1, 0, 1, ..., k}, e.g.. "Buy
10 shares of AAPL" or "Sell 10 shares of AAPL" are 10 or −10, respectively

* **Reward function r(s, a, s′)**: Reward is an incentive for an agent to learn a better policy. For example, it can be the change of the portfolio value when taking a at state s and arriving at new state s',  i.e., r(s, a, s′) = v′ − v, where v′ and v represent the portfolio values at state s′ and s, respectively


**Market environment**: 30 consituent stocks of Dow Jones Industrial Average (DJIA) index. Accessed at the starting date of the testing period.


The data for this case study is obtained from Yahoo Finance API. The data contains Open-High-Low-Close price and volume.


<a id='1'></a>
# Part 2. Install Python Packages

<a id='1.1'></a>
## 2.1. Install packages


In [4]:
## install required packages
!pip install swig
!pip install wrds
!pip install pyportfolioopt
## install finrl library
!pip install -q condacolab
# import condacolab
# condacolab.install()
!apt-get update -y -qq && apt-get install -y -qq cmake libopenmpi-dev python3-dev zlib1g-dev libgl1-mesa-glx swig
!pip install git+https://github.com/AI4Finance-Foundation/FinRL.git

zsh:1: command not found: apt-get
  Cloning https://github.com/AI4Finance-Foundation/FinRL.git to /private/var/folders/k3/x2ny3m2d3y57h8xp7cr2qxv80000gn/T/pip-req-build-9jr88x5m
  Running command git clone --filter=blob:none --quiet https://github.com/AI4Finance-Foundation/FinRL.git /private/var/folders/k3/x2ny3m2d3y57h8xp7cr2qxv80000gn/T/pip-req-build-9jr88x5m
  Resolved https://github.com/AI4Finance-Foundation/FinRL.git to commit e523ca0519288b3117f014e19163fe4c514d20ea
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Cloning https://github.com/AI4Finance-Foundation/ElegantRL.git to /private/var/folders/k3/x2ny3m2d3y57h8xp7cr2qxv80000gn/T/pip-install-s56z_rfy/elegantrl_efa04d3f43134eb8ae71d601cd546520
  Running command git clone --filter=blob:none --quiet https://github.com/AI4Finance-Foundation/ElegantRL.git /private/var/folders/k3/x2ny3m2d3y57h8xp7cr2qxv80000gn/T/pip-install-s56z_rfy/elegantrl_e


<a id='1.2'></a>
## 2.2. A list of Python packages 
* Yahoo Finance API
* pandas
* numpy
* matplotlib
* stockstats
* OpenAI gym
* stable-baselines
* tensorflow
* pyfolio

<a id='1.3'></a>
## 2.3. Import Packages

In [5]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
# matplotlib.use('Agg')
import datetime

%matplotlib inline
from finrl.meta.preprocessor.yahoodownloader import YahooDownloader
from finrl.meta.preprocessor.preprocessors import FeatureEngineer, data_split
from finrl.meta.env_stock_trading.env_stocktrading import StockTradingEnv
from finrl.agents.stablebaselines3.models import DRLAgent
from stable_baselines3.common.logger import configure
from finrl.meta.data_processor import DataProcessor

from finrl.plot import backtest_stats, backtest_plot, get_daily_return, get_baseline
from pprint import pprint

import sys
sys.path.append("../FinRL")

import itertools

/Users/pu17/miniconda3/envs/finrobot/lib/python3.10/site-packages/pyfolio/pos.py:26: UserWarning: Module "zipline.assets" not found; mutltipliers will not be applied to position notionals.
  warnings.warn(


<a id='1.4'></a>
## 2.4. Create Folders

In [6]:
from finrl import config
from finrl import config_tickers
import os
from finrl.main import check_and_make_directories
from finrl.config import (
    DATA_SAVE_DIR,
    TRAINED_MODEL_DIR,
    TENSORBOARD_LOG_DIR,
    RESULTS_DIR,
    INDICATORS,
    TRAIN_START_DATE,
    TRAIN_END_DATE,
    TEST_START_DATE,
    TEST_END_DATE,
    TRADE_START_DATE,
    TRADE_END_DATE,
)
check_and_make_directories([DATA_SAVE_DIR, TRAINED_MODEL_DIR, TENSORBOARD_LOG_DIR, RESULTS_DIR])



<a id='2'></a>
# Part 3. Download Data
Yahoo Finance provides stock data, financial news, financial reports, etc. Yahoo Finance is free.
* FinRL uses a class **YahooDownloader** in FinRL-Meta to fetch data via Yahoo Finance API
* Call Limit: Using the Public API (without authentication), you are limited to 2,000 requests per hour per IP (or up to a total of 48,000 requests a day).



-----
class YahooDownloader:
    Retrieving daily stock data from
    Yahoo Finance API

    Attributes
    ----------
        start_date : str
            start date of the data (modified from config.py)
        end_date : str
            end date of the data (modified from config.py)
        ticker_list : list
            a list of stock tickers (modified from config.py)

    Methods
    -------
    fetch_data()


In [7]:
# from config.py, TRAIN_START_DATE is a string
TRAIN_START_DATE
# from config.py, TRAIN_END_DATE is a string
TRAIN_END_DATE

'2020-07-31'

In [8]:
TRAIN_START_DATE = '2010-01-01'
TRAIN_END_DATE = '2022-10-01'
TRADE_START_DATE = '2022-10-01'
TRADE_END_DATE = '2024-10-22'

In [9]:
df = YahooDownloader(start_date = TRAIN_START_DATE,
                     end_date = TRADE_END_DATE,
                     ticker_list = config_tickers.DOW_30_TICKER).fetch_data()

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%********

Shape of DataFrame:  (109433, 8)


In [10]:
print(config_tickers.DOW_30_TICKER)

['AXP', 'AMGN', 'AAPL', 'BA', 'CAT', 'CSCO', 'CVX', 'GS', 'HD', 'HON', 'IBM', 'INTC', 'JNJ', 'KO', 'JPM', 'MCD', 'MMM', 'MRK', 'MSFT', 'NKE', 'PG', 'TRV', 'UNH', 'CRM', 'VZ', 'V', 'WBA', 'WMT', 'DIS', 'DOW']


In [11]:
df.shape

(109433, 8)

In [12]:
df.sort_values(['date','tic'],ignore_index=True).head()

,date,open,high,low,close,volume,tic,day
0,2010-01-04,7.622500,7.660714,7.585000,6.454505,493729600,AAPL,0
1,2010-01-04,56.630001,57.869999,56.560001,40.915894,5277400,AMGN,0
2,2010-01-04,40.810001,41.099998,40.389999,32.906181,6894300,AXP,0
3,2010-01-04,55.720001,56.389999,54.799999,43.777546,6186700,BA,0
4,2010-01-04,57.650002,59.189999,57.509998,39.883923,7325600,CAT,0


# Part 4: Preprocess Data
We need to check for missing data and do feature engineering to convert the data point into a state.
* **Adding technical indicators**. In practical trading, various information needs to be taken into account, such as historical prices, current holding shares, technical indicators, etc. Here, we demonstrate two trend-following technical indicators: MACD and RSI.
* **Adding turbulence index**. Risk-aversion reflects whether an investor prefers to protect the capital. It also influences one's trading strategy when facing different market volatility level. To control the risk in a worst-case scenario, such as financial crisis of 2007–2008, FinRL employs the turbulence index that measures extreme fluctuation of asset price.

In [13]:
fe = FeatureEngineer(
                    use_technical_indicator=True,
                    tech_indicator_list = INDICATORS,
                    use_vix=True,
                    use_turbulence=True,
                    user_defined_feature = False)

processed = fe.preprocess_data(df)

Successfully added technical indicators


[*********************100%***********************]  1 of 1 completed


Shape of DataFrame:  (3724, 8)
Successfully added vix
Successfully added turbulence index


In [14]:
list_ticker = processed["tic"].unique().tolist()
list_date = list(pd.date_range(processed['date'].min(),processed['date'].max()).astype(str))
combination = list(itertools.product(list_date,list_ticker))

processed_full = pd.DataFrame(combination,columns=["date","tic"]).merge(processed,on=["date","tic"],how="left")
processed_full = processed_full[processed_full['date'].isin(processed['date'])]
processed_full = processed_full.sort_values(['date','tic'])

processed_full = processed_full.fillna(0)

In [15]:
processed_full.sort_values(['date','tic'],ignore_index=True).head(10)

,date,tic,open,high,low,close,volume,day,macd,boll_ub,boll_lb,rsi_30,cci_30,dx_30,close_30_sma,close_60_sma,vix,turbulence
0,2010-01-04,AAPL,7.622500,7.660714,7.585000,6.454505,493729600.0,0.0,0.0,6.475865,6.444304,100.0,66.666667,100.0,6.454505,6.454505,20.040001,0.0
1,2010-01-04,AMGN,56.630001,57.869999,56.560001,40.915894,5277400.0,0.0,0.0,6.475865,6.444304,100.0,66.666667,100.0,40.915894,40.915894,20.040001,0.0
2,2010-01-04,AXP,40.810001,41.099998,40.389999,32.906181,6894300.0,0.0,0.0,6.475865,6.444304,100.0,66.666667,100.0,32.906181,32.906181,20.040001,0.0
3,2010-01-04,BA,55.720001,56.389999,54.799999,43.777546,6186700.0,0.0,0.0,6.475865,6.444304,100.0,66.666667,100.0,43.777546,43.777546,20.040001,0.0
4,2010-01-04,CAT,57.650002,59.189999,57.509998,39.883923,7325600.0,0.0,0.0,6.475865,6.444304,100.0,66.666667,100.0,39.883923,39.883923,20.040001,0.0
5,2010-01-04,CRM,18.652500,18.882500,18.547501,18.622328,7906000.0,0.0,0.0,6.475865,6.444304,100.0,66.666667,100.0,18.622328,18.622328,20.040001,0.0
6,2010-01-04,CSCO,24.110001,24.840000,24.010000,16.475374,59853700.0,0.0,0.0,6.475865,6.444304,100.0,66.666667,100.0,16.475374,16.475374,20.040001,0.0
7,2010-01-04,CVX,78.199997,79.199997,78.160004,44.045547,10173800.0,0.0,0.0,6.475865,6.444304,100.0,66.666667,100.0,44.045547,44.045547,20.040001,0.0
8,2010-01-04,DIS,32.500000,32.750000,31.870001,27.715307,13700400.0,0.0,0.0,6.475865,6.444304,100.0,66.666667,100.0,27.715307,27.715307,20.040001,0.0
9,2010-01-04,GS,170.050003,174.250000,169.509995,133.968094,9135000.0,0.0,0.0,6.475865,6.444304,100.0,66.666667,100.0,133.968094,133.968094,20.040001,0.0


In [16]:
mvo_df = processed_full.sort_values(['date','tic'],ignore_index=True)[['date','tic','close']]

<a id='4'></a>
# Part 5. Build A Market Environment in OpenAI Gym-style
The training process involves observing stock price change, taking an action and reward's calculation. By interacting with the market environment, the agent will eventually derive a trading strategy that may maximize (expected) rewards.

Our market environment, based on OpenAI Gym, simulates stock markets with historical market data.

## Data Split
We split the data into training set and testing set as follows:

Training data period: 2009-01-01 to 2020-07-01

Trading data period: 2020-07-01 to 2021-10-31


In [17]:
train = data_split(processed_full, TRAIN_START_DATE,TRAIN_END_DATE)
trade = data_split(processed_full, TRADE_START_DATE,TRADE_END_DATE)
train_length = len(train)
trade_length = len(trade)
print(train_length)
print(trade_length)

93061
14935


In [18]:
train.tail()

,date,tic,open,high,low,close,volume,day,macd,boll_ub,boll_lb,rsi_30,cci_30,dx_30,close_30_sma,close_60_sma,vix,turbulence
3208,2022-09-30,UNH,511.100006,516.549988,504.839996,490.247040,3116000.0,4.0,-4.766460,515.237186,487.893204,45.416875,-108.388997,11.112651,506.361638,510.073797,31.620001,78.864327
3208,2022-09-30,V,180.059998,182.919998,177.600006,174.855118,9472300.0,4.0,-6.977798,207.377722,170.032752,34.518088,-134.449018,53.645954,193.067018,200.313066,31.620001,78.864327
3208,2022-09-30,VZ,38.540001,38.790001,37.950001,32.612579,23523600.0,4.0,-1.154057,37.008024,32.536550,30.404971,-156.481654,59.319592,35.552029,37.863195,31.620001,78.864327
3208,2022-09-30,WBA,31.660000,31.790001,31.350000,27.575823,8035300.0,4.0,-1.169420,32.647102,27.290797,31.991897,-159.623014,44.485690,30.634926,32.268419,31.620001,78.864327
3208,2022-09-30,WMT,44.080002,44.119999,43.233334,42.014282,19602300.0,4.0,-0.163221,44.725694,41.850297,47.024202,-142.943026,18.334479,43.332266,42.651682,31.620001,78.864327


In [19]:
trade.head()

,date,tic,open,high,low,close,volume,day,macd,boll_ub,boll_lb,rsi_30,cci_30,dx_30,close_30_sma,close_60_sma,vix,turbulence
0,2022-10-03,AAPL,138.210007,143.070007,137.690002,140.888962,114311700.0,0.0,-4.344756,161.704234,139.136487,41.887032,-179.457878,30.803913,153.927834,156.044644,30.1,28.650693
0,2022-10-03,AMGN,228.270004,230.949997,226.419998,216.426392,2521700.0,0.0,-3.582168,231.593610,203.934917,44.973430,-57.081979,19.039144,221.214680,226.329626,30.1,28.650693
0,2022-10-03,AXP,137.949997,140.770004,134.919998,135.828201,2677100.0,0.0,-4.708902,160.270220,127.370887,42.415962,-127.411821,30.776646,146.187831,147.552545,30.1,28.650693
0,2022-10-03,BA,122.000000,127.150002,121.019997,126.050003,7962400.0,0.0,-8.773702,166.337388,117.446611,38.629973,-140.965482,31.178749,148.258667,154.151834,30.1,28.650693
0,2022-10-03,CAT,167.429993,173.369995,166.050003,164.048492,3444700.0,0.0,-5.538078,186.496137,150.582695,43.702394,-83.572754,5.658073,173.056826,175.188273,30.1,28.650693


In [20]:
INDICATORS

['macd',
 'boll_ub',
 'boll_lb',
 'rsi_30',
 'cci_30',
 'dx_30',
 'close_30_sma',
 'close_60_sma']

In [21]:
stock_dimension = len(train.tic.unique())
state_space = 1 + 2*stock_dimension + len(INDICATORS)*stock_dimension
print(f"Stock Dimension: {stock_dimension}, State Space: {state_space}")

Stock Dimension: 29, State Space: 291


In [22]:
buy_cost_list = sell_cost_list = [0.001] * stock_dimension
num_stock_shares = [0] * stock_dimension

env_kwargs = {
    "hmax": 100,
    "initial_amount": 1000000,
    "num_stock_shares": num_stock_shares,
    "buy_cost_pct": buy_cost_list,
    "sell_cost_pct": sell_cost_list,
    "state_space": state_space,
    "stock_dim": stock_dimension,
    "tech_indicator_list": INDICATORS,
    "action_space": stock_dimension,
    "reward_scaling": 1e-4
}


e_train_gym = StockTradingEnv(df = train, **env_kwargs)

## Environment for Training



In [23]:
env_train, _ = e_train_gym.get_sb_env()
print(type(env_train))

<class 'stable_baselines3.common.vec_env.dummy_vec_env.DummyVecEnv'>


<a id='5'></a>
# Part 6: Train DRL Agents
* The DRL algorithms are from **Stable Baselines 3**. Users are also encouraged to try **ElegantRL** and **Ray RLlib**.
* FinRL includes fine-tuned standard DRL algorithms, such as DQN, DDPG, Multi-Agent DDPG, PPO, SAC, A2C and TD3. We also allow users to
design their own DRL algorithms by adapting these DRL algorithms.

In [24]:
agent = DRLAgent(env = env_train)

if_using_a2c = True
if_using_ddpg = True
if_using_ppo = True
if_using_td3 = True
if_using_sac = True


### Agent Training: 5 algorithms (A2C, DDPG, PPO, TD3, SAC)


### Agent 1: A2C


In [25]:
agent = DRLAgent(env = env_train)
model_a2c = agent.get_model("a2c")

if if_using_a2c:
  # set up logger
  tmp_path = RESULTS_DIR + '/a2c'
  new_logger_a2c = configure(tmp_path, ["stdout", "csv", "tensorboard"])
  # Set new logger
  model_a2c.set_logger(new_logger_a2c)


{'n_steps': 5, 'ent_coef': 0.01, 'learning_rate': 0.0007}
Using cpu device
Logging to results/a2c


In [26]:
trained_a2c = agent.train_model(model=model_a2c, 
                             tb_log_name='a2c',
                             total_timesteps=50000) if if_using_a2c else None

--------------------------------------
| time/                 |            |
|    fps                | 203        |
|    iterations         | 100        |
|    time_elapsed       | 2          |
|    total_timesteps    | 500        |
| train/                |            |
|    entropy_loss       | -41.2      |
|    explained_variance | 0.464      |
|    learning_rate      | 0.0007     |
|    n_updates          | 99         |
|    policy_loss        | -43        |
|    reward             | 0.48319802 |
|    std                | 1          |
|    value_loss         | 1.43       |
--------------------------------------
---------------------------------------
| time/                 |             |
|    fps                | 188         |
|    iterations         | 200         |
|    time_elapsed       | 5           |
|    total_timesteps    | 1000        |
| train/                |             |
|    entropy_loss       | -41.2       |
|    explained_variance | 0           |
|    learning_ra

### Agent 2: DDPG

In [27]:
agent = DRLAgent(env = env_train)
model_ddpg = agent.get_model("ddpg")

if if_using_ddpg:
  # set up logger
  tmp_path = RESULTS_DIR + '/ddpg'
  new_logger_ddpg = configure(tmp_path, ["stdout", "csv", "tensorboard"])
  # Set new logger
  model_ddpg.set_logger(new_logger_ddpg)

{'batch_size': 128, 'buffer_size': 50000, 'learning_rate': 0.001}
Using cpu device
Logging to results/ddpg


In [28]:
trained_ddpg = agent.train_model(model=model_ddpg, 
                             tb_log_name='ddpg',
                             total_timesteps=50000) if if_using_ddpg else None

day: 3208, episode: 20
begin_total_asset: 1000000.00
end_total_asset: 5732024.96
total_reward: 4732024.96
total_cost: 1184.92
total_trades: 48120
Sharpe: 0.940
----------------------------------
| time/              |           |
|    episodes        | 4         |
|    fps             | 72        |
|    time_elapsed    | 176       |
|    total_timesteps | 12836     |
| train/             |           |
|    actor_loss      | 3.6       |
|    critic_loss     | 9.24      |
|    learning_rate   | 0.001     |
|    n_updates       | 12735     |
|    reward          | -10.58833 |
----------------------------------
----------------------------------
| time/              |           |
|    episodes        | 8         |
|    fps             | 76        |
|    time_elapsed    | 337       |
|    total_timesteps | 25672     |
| train/             |           |
|    actor_loss      | -7.66     |
|    critic_loss     | 7.81      |
|    learning_rate   | 0.001     |
|    n_updates       | 25571     |


### Agent 3: PPO

In [29]:
agent = DRLAgent(env = env_train)
PPO_PARAMS = {
    "n_steps": 2048,
    "ent_coef": 0.01,
    "learning_rate": 0.00025,
    "batch_size": 128,
}
model_ppo = agent.get_model("ppo",model_kwargs = PPO_PARAMS)

if if_using_ppo:
  # set up logger
  tmp_path = RESULTS_DIR + '/ppo'
  new_logger_ppo = configure(tmp_path, ["stdout", "csv", "tensorboard"])
  # Set new logger
  model_ppo.set_logger(new_logger_ppo)

{'n_steps': 2048, 'ent_coef': 0.01, 'learning_rate': 0.00025, 'batch_size': 128}
Using cpu device
Logging to results/ppo


In [30]:
trained_ppo = agent.train_model(model=model_ppo, 
                             tb_log_name='ppo',
                             total_timesteps=50000) if if_using_ppo else None

----------------------------------
| time/              |           |
|    fps             | 222       |
|    iterations      | 1         |
|    time_elapsed    | 9         |
|    total_timesteps | 2048      |
| train/             |           |
|    reward          | 1.9766332 |
----------------------------------
-----------------------------------------
| time/                   |             |
|    fps                  | 228         |
|    iterations           | 2           |
|    time_elapsed         | 17          |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.016784824 |
|    clip_fraction        | 0.205       |
|    clip_range           | 0.2         |
|    entropy_loss         | -41.2       |
|    explained_variance   | -0.0104     |
|    learning_rate        | 0.00025     |
|    loss                 | 5.61        |
|    n_updates            | 10          |
|    policy_gradient_loss | -0.0187     |
|    reward  

### Agent 4: TD3

In [31]:
agent = DRLAgent(env = env_train)
TD3_PARAMS = {"batch_size": 100, 
              "buffer_size": 1000000, 
              "learning_rate": 0.001}

model_td3 = agent.get_model("td3",model_kwargs = TD3_PARAMS)

if if_using_td3:
  # set up logger
  tmp_path = RESULTS_DIR + '/td3'
  new_logger_td3 = configure(tmp_path, ["stdout", "csv", "tensorboard"])
  # Set new logger
  model_td3.set_logger(new_logger_td3)

{'batch_size': 100, 'buffer_size': 1000000, 'learning_rate': 0.001}
Using cpu device
Logging to results/td3


In [32]:
trained_td3 = agent.train_model(model=model_td3, 
                             tb_log_name='td3',
                             total_timesteps=50000) if if_using_td3 else None

day: 3208, episode: 50
begin_total_asset: 1000000.00
end_total_asset: 3992906.61
total_reward: 2992906.61
total_cost: 6364.11
total_trades: 43426
Sharpe: 0.710
----------------------------------
| time/              |           |
|    episodes        | 4         |
|    fps             | 31        |
|    time_elapsed    | 412       |
|    total_timesteps | 12836     |
| train/             |           |
|    actor_loss      | -29.4     |
|    critic_loss     | 23.1      |
|    learning_rate   | 0.001     |
|    n_updates       | 12735     |
|    reward          | -9.579912 |
----------------------------------
----------------------------------
| time/              |           |
|    episodes        | 8         |
|    fps             | 45        |
|    time_elapsed    | 567       |
|    total_timesteps | 25672     |
| train/             |           |
|    actor_loss      | -20.6     |
|    critic_loss     | 49.4      |
|    learning_rate   | 0.001     |
|    n_updates       | 25571     |


### Agent 5: SAC

In [33]:
agent = DRLAgent(env = env_train)
SAC_PARAMS = {
    "batch_size": 128,
    "buffer_size": 100000,
    "learning_rate": 0.0001,
    "learning_starts": 100,
    "ent_coef": "auto_0.1",
}

model_sac = agent.get_model("sac",model_kwargs = SAC_PARAMS)

if if_using_sac:
  # set up logger
  tmp_path = RESULTS_DIR + '/sac'
  new_logger_sac = configure(tmp_path, ["stdout", "csv", "tensorboard"])
  # Set new logger
  model_sac.set_logger(new_logger_sac)

{'batch_size': 128, 'buffer_size': 100000, 'learning_rate': 0.0001, 'learning_starts': 100, 'ent_coef': 'auto_0.1'}
Using cpu device
Logging to results/sac


In [34]:
trained_sac = agent.train_model(model=model_sac, 
                             tb_log_name='sac',
                             total_timesteps=50000) if if_using_sac else None

-----------------------------------
| time/              |            |
|    episodes        | 4          |
|    fps             | 70         |
|    time_elapsed    | 182        |
|    total_timesteps | 12836      |
| train/             |            |
|    actor_loss      | 818        |
|    critic_loss     | 194        |
|    ent_coef        | 0.134      |
|    ent_coef_loss   | -86.1      |
|    learning_rate   | 0.0001     |
|    n_updates       | 12735      |
|    reward          | -7.6616063 |
-----------------------------------
day: 3208, episode: 70
begin_total_asset: 1000000.00
end_total_asset: 3099968.14
total_reward: 2099968.14
total_cost: 28125.57
total_trades: 59172
Sharpe: 0.467
-----------------------------------
| time/              |            |
|    episodes        | 8          |
|    fps             | 71         |
|    time_elapsed    | 359        |
|    total_timesteps | 25672      |
| train/             |            |
|    actor_loss      | 368        |
|    critic

## In-sample Performance

Assume that the initial capital is $1,000,000.

### Set turbulence threshold
Set the turbulence threshold to be greater than the maximum of insample turbulence data. If current turbulence index is greater than the threshold, then we assume that the current market is volatile

In [35]:
data_risk_indicator = processed_full[(processed_full.date<TRAIN_END_DATE) & (processed_full.date>=TRAIN_START_DATE)]
insample_risk_indicator = data_risk_indicator.drop_duplicates(subset=['date'])

In [36]:
insample_risk_indicator.vix.describe()

count    3209.000000
mean       18.579857
std         7.304794
min         9.140000
25%        13.560000
50%        16.650000
75%        21.530001
max        82.690002
Name: vix, dtype: float64

In [37]:
insample_risk_indicator.vix.quantile(0.996)

57.063361450195316

In [38]:
insample_risk_indicator.turbulence.describe()

count    3209.000000
mean       34.999091
std        43.504660
min         0.000000
25%        14.973016
50%        24.198460
75%        39.785822
max       652.500239
Name: turbulence, dtype: float64

In [39]:
insample_risk_indicator.turbulence.quantile(0.996)

276.7862276045582

### Trading (Out-of-sample Performance)

We update periodically in order to take full advantage of the data, e.g., retrain quarterly, monthly or weekly. We also tune the parameters along the way, in this notebook we use the in-sample data from 2009-01 to 2020-07 to tune the parameters once, so there is some alpha decay here as the length of trade date extends. 

Numerous hyperparameters – e.g. the learning rate, the total number of samples to train on – influence the learning process and are usually determined by testing some variations.

In [40]:
e_trade_gym = StockTradingEnv(df = trade, turbulence_threshold = 70,risk_indicator_col='vix', **env_kwargs)
# env_trade, obs_trade = e_trade_gym.get_sb_env()

In [41]:
trade.head()

,date,tic,open,high,low,close,volume,day,macd,boll_ub,boll_lb,rsi_30,cci_30,dx_30,close_30_sma,close_60_sma,vix,turbulence
0,2022-10-03,AAPL,138.210007,143.070007,137.690002,140.888962,114311700.0,0.0,-4.344756,161.704234,139.136487,41.887032,-179.457878,30.803913,153.927834,156.044644,30.1,28.650693
0,2022-10-03,AMGN,228.270004,230.949997,226.419998,216.426392,2521700.0,0.0,-3.582168,231.593610,203.934917,44.973430,-57.081979,19.039144,221.214680,226.329626,30.1,28.650693
0,2022-10-03,AXP,137.949997,140.770004,134.919998,135.828201,2677100.0,0.0,-4.708902,160.270220,127.370887,42.415962,-127.411821,30.776646,146.187831,147.552545,30.1,28.650693
0,2022-10-03,BA,122.000000,127.150002,121.019997,126.050003,7962400.0,0.0,-8.773702,166.337388,117.446611,38.629973,-140.965482,31.178749,148.258667,154.151834,30.1,28.650693
0,2022-10-03,CAT,167.429993,173.369995,166.050003,164.048492,3444700.0,0.0,-5.538078,186.496137,150.582695,43.702394,-83.572754,5.658073,173.056826,175.188273,30.1,28.650693


In [42]:
trained_moedl = trained_a2c
df_account_value_a2c, df_actions_a2c = DRLAgent.DRL_prediction(
    model=trained_moedl, 
    environment = e_trade_gym)

hit end!


In [43]:
trained_moedl = trained_ddpg
df_account_value_ddpg, df_actions_ddpg = DRLAgent.DRL_prediction(
    model=trained_moedl, 
    environment = e_trade_gym)

hit end!


In [44]:
trained_moedl = trained_ppo
df_account_value_ppo, df_actions_ppo = DRLAgent.DRL_prediction(
    model=trained_moedl, 
    environment = e_trade_gym)

hit end!


In [45]:
trained_moedl = trained_td3
df_account_value_td3, df_actions_td3 = DRLAgent.DRL_prediction(
    model=trained_moedl, 
    environment = e_trade_gym)
print(df_account_value_td3, df_actions_td3)

hit end!
           date  account_value
0    2022-10-03   1.000000e+06
1    2022-10-04   1.006827e+06
2    2022-10-05   1.006407e+06
3    2022-10-06   9.970012e+05
4    2022-10-07   9.727281e+05
..          ...            ...
510  2024-10-14   1.513660e+06
511  2024-10-15   1.494411e+06
512  2024-10-16   1.510790e+06
513  2024-10-17   1.510509e+06
514  2024-10-18   1.511075e+06

[515 rows x 2 columns]             AAPL  AMGN  AXP  BA  CAT  CRM  CSCO  CVX  DIS   GS  ...  MRK  \
date                                                            ...        
2022-10-03     0     0  100   0    0  100     0    0    0  100  ...    0   
2022-10-04     0     0  100   0    0  100     0    0    0  100  ...    0   
2022-10-05     0     0  100   0    0  100     0    0    0  100  ...    0   
2022-10-06     0     0  100   0    0  100     0    0    0  100  ...    0   
2022-10-07     0     0    0   0    0    0     0    0    0    0  ...    0   
...          ...   ...  ...  ..  ...  ...   ...  ...  ...  ... 

In [46]:
trained_moedl = trained_sac
df_account_value_sac, df_actions_sac = DRLAgent.DRL_prediction(
    model=trained_moedl, 
    environment = e_trade_gym)

hit end!


In [47]:
df_account_value_a2c.shape

(515, 2)

<a id='7'></a>
# Part 6.5: Mean Variance Optimization

Mean Variance optimization is a very classic strategy in portfolio management. Here, we go through the whole process to do the mean variance optimization and add it as a baseline to compare.

First, process dataframe to the form for MVO weight calculation.

In [48]:
def process_df_for_mvo(df):
  df = df.sort_values(['date','tic'],ignore_index=True)[['date','tic','close']]
  fst = df
  fst = fst.iloc[0:stock_dimension, :]
  tic = fst['tic'].tolist()

  mvo = pd.DataFrame()

  for k in range(len(tic)):
    mvo[tic[k]] = 0

  for i in range(df.shape[0]//stock_dimension):
    n = df
    n = n.iloc[i * stock_dimension:(i+1) * stock_dimension, :]
    date = n['date'][i*stock_dimension]
    mvo.loc[date] = n['close'].tolist()
  
  return mvo

### Helper functions for mean returns and variance-covariance matrix

In [49]:
# Codes in this section partially refer to Dr G A Vijayalakshmi Pai

# https://www.kaggle.com/code/vijipai/lesson-5-mean-variance-optimization-of-portfolios/notebook

def StockReturnsComputing(StockPrice, Rows, Columns): 
  import numpy as np 
  StockReturn = np.zeros([Rows-1, Columns]) 
  for j in range(Columns):        # j: Assets 
    for i in range(Rows-1):     # i: Daily Prices 
      StockReturn[i,j]=((StockPrice[i+1, j]-StockPrice[i,j])/StockPrice[i,j])* 100 
      
  return StockReturn

### Calculate the weights for mean-variance

In [50]:
train_mvo = data_split(processed_full, TRAIN_START_DATE,TRAIN_END_DATE).reset_index()
trade_mvo = data_split(processed_full, TRADE_START_DATE,TRADE_END_DATE).reset_index()

In [51]:
StockData = process_df_for_mvo(train_mvo)
TradeData = process_df_for_mvo(trade_mvo)

TradeData.to_numpy()

array([[140.88896179, 216.4263916 , 135.82820129, ...,  33.63466263,
         28.48038101,  42.93101883],
       [144.49897766, 218.8494873 , 141.1053772 , ...,  34.19296265,
         29.32346535,  43.48818588],
       [144.79568481, 219.6008606 , 140.04795837, ...,  33.84081268,
         29.12147713,  43.05735779],
       ...,
       [231.77999878, 321.63000488, 281.67999268, ...,  43.90999985,
         11.06999969,  81.22000122],
       [232.1499939 , 321.32998657, 285.77999878, ...,  43.84999847,
         10.65999985,  80.88999939],
       [235.        , 321.66000366, 276.79000854, ...,  43.99000168,
         10.78999996,  81.30999756]])

In [52]:
#compute asset returns
arStockPrices = np.asarray(StockData)
[Rows, Cols]=arStockPrices.shape
arReturns = StockReturnsComputing(arStockPrices, Rows, Cols)

#compute mean returns and variance covariance matrix of returns
meanReturns = np.mean(arReturns, axis = 0)
covReturns = np.cov(arReturns, rowvar=False)
 
#set precision for printing results
np.set_printoptions(precision=3, suppress = True)

#display mean returns and variance-covariance matrix of returns
print('Mean returns of assets in k-portfolio 1\n', meanReturns)
print('Variance-Covariance matrix of returns\n', covReturns)

Mean returns of assets in k-portfolio 1
 [0.111 0.063 0.06  0.058 0.06  0.09  0.04  0.049 0.051 0.04  0.091 0.065
 0.022 0.036 0.046 0.054 0.039 0.059 0.029 0.05  0.084 0.07  0.041 0.055
 0.105 0.081 0.031 0.021 0.044]
Variance-Covariance matrix of returns
 [[3.211 0.997 1.392 1.655 1.375 1.849 1.428 1.105 1.199 1.43  1.228 1.258
  1.032 1.612 0.687 1.315 0.691 0.853 1.053 0.722 1.697 1.299 0.695 0.851
  1.17  1.419 0.531 0.964 0.657]
 [0.997 2.341 1.056 0.975 1.032 1.167 1.004 0.903 0.91  1.092 0.956 1.004
  0.829 1.133 0.88  1.106 0.642 0.655 0.898 1.03  1.062 0.855 0.716 0.832
  1.117 1.03  0.628 1.018 0.621]
 [1.392 1.056 3.446 2.611 1.946 1.687 1.468 1.878 1.807 2.28  1.404 1.855
  1.341 1.58  0.849 2.405 1.006 1.096 1.395 0.921 1.435 1.547 0.74  1.518
  1.411 1.892 0.734 1.252 0.583]
 [1.655 0.975 2.611 5.251 2.172 1.854 1.544 2.099 1.999 2.269 1.559 2.146
  1.509 1.827 0.851 2.328 1.106 1.212 1.482 0.88  1.519 1.737 0.743 1.57
  1.445 1.815 0.733 1.431 0.6  ]
 [1.375 1.032 1.946

### Use PyPortfolioOpt

In [53]:
from pypfopt.efficient_frontier import EfficientFrontier

ef_mean = EfficientFrontier(meanReturns, covReturns, weight_bounds=(0, 0.5))
raw_weights_mean = ef_mean.max_sharpe()
cleaned_weights_mean = ef_mean.clean_weights()
mvo_weights = np.array([1000000 * cleaned_weights_mean[i] for i in range(29)])
mvo_weights

array([328880.,      0.,      0.,      0.,      0.,      0.,      0.,
            0.,      0.,      0., 267170.,      0.,      0.,      0.,
            0.,      0.,      0.,      0.,      0.,      0.,      0.,
            0.,      0.,      0., 403960.,      0.,      0.,      0.,
            0.])

In [54]:
LastPrice = np.array([1/p for p in StockData.tail(1).to_numpy()[0]])
Initial_Portfolio = np.multiply(mvo_weights, LastPrice)
Initial_Portfolio

array([2406.107,    0.   ,    0.   ,    0.   ,    0.   ,    0.   ,
          0.   ,    0.   ,    0.   ,    0.   , 1020.33 ,    0.   ,
          0.   ,    0.   ,    0.   ,    0.   ,    0.   ,    0.   ,
          0.   ,    0.   ,    0.   ,    0.   ,    0.   ,    0.   ,
        823.993,    0.   ,    0.   ,    0.   ,    0.   ])

In [55]:
Portfolio_Assets = TradeData @ Initial_Portfolio
MVO_result = pd.DataFrame(Portfolio_Assets, columns=["Mean Var"])
# MVO_result

<a id='6'></a>
# Part 7: Backtesting Results
Backtesting plays a key role in evaluating the performance of a trading strategy. Automated backtesting tool is preferred because it reduces the human error. We usually use the Quantopian pyfolio package to backtest our trading strategies. It is easy to use and consists of various individual plots that provide a comprehensive image of the performance of a trading strategy.

In [56]:
df_result_a2c = df_account_value_a2c.set_index(df_account_value_a2c.columns[0])
df_result_a2c.rename(columns = {'account_value':'a2c'}, inplace = True)
df_result_ddpg = df_account_value_ddpg.set_index(df_account_value_ddpg.columns[0])
df_result_ddpg.rename(columns = {'account_value':'ddpg'}, inplace = True)
df_result_td3 = df_account_value_td3.set_index(df_account_value_td3.columns[0])
df_result_td3.rename(columns = {'account_value':'td3'}, inplace = True)
df_result_ppo = df_account_value_ppo.set_index(df_account_value_ppo.columns[0])
df_result_ppo.rename(columns = {'account_value':'ppo'}, inplace = True)
df_result_sac = df_account_value_sac.set_index(df_account_value_sac.columns[0])
df_result_sac.rename(columns = {'account_value':'sac'}, inplace = True)
df_account_value_a2c.to_csv("df_account_value_a2c.csv")
#baseline stats
print("==============Get Baseline Stats===========")
df_dji_ = get_baseline(
        ticker="^DJI", 
        start = TRADE_START_DATE,
        end = TRADE_END_DATE)
stats = backtest_stats(df_dji_, value_col_name = 'close')
df_dji = pd.DataFrame()
df_dji['date'] = df_account_value_a2c['date']
df_dji['account_value'] = df_dji_['close'] / df_dji_['close'][0] * env_kwargs["initial_amount"]
df_dji.to_csv("df_dji.csv")
df_dji = df_dji.set_index(df_dji.columns[0])
df_dji.to_csv("df_dji+.csv")

result = pd.DataFrame()
result = pd.merge(result, df_result_a2c, how='outer', left_index=True, right_index=True)
result = pd.merge(result, df_result_ddpg, how='outer', left_index=True, right_index=True)
result = pd.merge(result, df_result_td3, how='outer', left_index=True, right_index=True)
result = pd.merge(result, df_result_ppo, how='outer', left_index=True, right_index=True)
result = pd.merge(result, df_result_sac, how='outer', left_index=True, right_index=True)
result = pd.merge(result, MVO_result, how='outer', left_index=True, right_index=True)
print(result.head())
result = pd.merge(result, df_dji, how='outer', left_index=True, right_index=True)
# result.columns = ['a2c', 'ddpg', 'td3', 'ppo', 'sac', 'mean var', 'dji']

# print("result: ", result)
result.to_csv("result.csv")

==============Get Baseline Stats===========


[*********************100%***********************]  1 of 1 completed

Shape of DataFrame:  (516, 8)
Annual return          0.201291
Cumulative returns     0.455758
Annual volatility      0.124837
Sharpe ratio           1.534657
Calmar ratio           2.232161
Stability              0.859189
Max drawdown          -0.090178
Omega ratio            1.296442
Sortino ratio          2.382899
Skew                        NaN
Kurtosis                    NaN
Tail ratio             1.053362
Daily value at risk   -0.014968
dtype: float64
                     a2c          ddpg           td3           ppo  \
date                                                                 
2022-10-03  1.000000e+06  1.000000e+06  1.000000e+06  1.000000e+06   
2022-10-04  1.005003e+06  1.004230e+06  1.006827e+06  1.000787e+06   
2022-10-05  1.004617e+06  1.004971e+06  1.006407e+06  1.000548e+06   
2022-10-06  9.997465e+05  9.967405e+05  9.970012e+05  9.996065e+05   
2022-10-07  9.855529e+05  9.774952e+05  9.727281e+05  9.968496e+05   

                     sac      Mean Var  
date   

In [57]:
df_result_ddpg

,ddpg
date,
2022-10-03,1.000000e+06
2022-10-04,1.004230e+06
2022-10-05,1.004971e+06
2022-10-06,9.967405e+05
2022-10-07,9.774952e+05
...,...
2024-10-14,1.393027e+06
2024-10-15,1.377365e+06
2024-10-16,1.385581e+06


In [59]:
%matplotlib inline
plt.rcParams["figure.figsize"] = (15,5)
plt.figure();
result.plot();